<a href="https://colab.research.google.com/github/NomadZhang/DSA5204/blob/main/training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/NomadZhang/DSA5204.git

%cd DSA5204

!pip install torch transformers datasets accelerate

Cloning into 'DSA5204'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 30 (delta 6), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 4.66 MiB | 12.80 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/DSA5204


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from src.common import get_device
from src.lora import inject_lora

device = get_device()
print(f"Current cloud-based devices: {device}")

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the model and tokenizer
print("Model and tokenizer is being loaded...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)

for param in model.parameters():
    param.requires_grad = False
model = inject_lora(model, target_modules=("q_proj", "v_proj"), r=8, alpha=16)

model.to(device)
print("Lora injected! Everything is ready!")

Current cloud-based devices: cuda
Model and tokenizer is being loaded...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Lora injected! Everything is ready!


In [4]:
# Load the dataset
dataset = load_dataset('json', data_files='./data/train.jsonl')

def tokenize_function(examples):
    # Transform text into Token IDs
    texts = [p + r for p, r in zip(examples['prompt'], examples['response'])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=256)

print("Dataset is being processed...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset has been processed!")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset is being processed...


Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset has been processed!


In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Tell model how to learn
training_args = TrainingArguments(
    output_dir="./lora_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=True,
)

# Auto-calculate the loss
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

print("🚀 Light the fire! Start the training...")
trainer.train()

# Save *only* the LoRA adapter weights and config
model.save_pretrained("./tinyllama-lora-finetuned")
print("🎉 Training completed!")

🚀 Light the fire! Start the training...


Step,Training Loss
10,2.137772
20,2.011255
30,1.918126
40,1.907751
50,1.881153
60,1.827230
70,1.839773
80,1.808788
90,1.783735
100,1.903173


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Training completed!
